In [1]:
import os
from cerebras.cloud.sdk import Cerebras
import pandas as pd
import numpy as np
import json
from config_gen import generate_coalitional_conf, generate_divergent_conf, generate_minorty_conf, generate_uniform_conf
from strats import ADD, MPL, APP, LMS, MAJ, FAI,MAJ_from_df, BORDA, AWM, BORDA_from_df
import random
import time
import re
from ollama import Client
from cerebras.cloud.sdk import RateLimitError


In [2]:
client_ol = Client() #requires ollama logged in on device (mac). Pipeline, however, is adaptable for e.g, Huggingface. Only the LLM call needs to be switched.


### Helper functions which are needed further down


def dcg_at_k(relevance_scores, k=10):
    relevance_scores = np.array(relevance_scores)[:k]
    return np.sum(relevance_scores / np.log2(np.arange(2, len(relevance_scores) + 2)))

def ndcg_at_k(predicted_order, gold_order, k=10, binary_relevance=False):

    if binary_relevance:
        gold_set = set(gold_order)
        predicted_relevance = [1 if item in gold_set else 0 for item in predicted_order]
        ideal_relevance = [1] * len(gold_order)  
    else:
        relevance_map = {item: len(gold_order) - i for i, item in enumerate(gold_order)}
        predicted_relevance = [relevance_map.get(item, 0) for item in predicted_order]
        ideal_relevance = [relevance_map[item] for item in gold_order]
    
    dcg = dcg_at_k(predicted_relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg



In [3]:
## Initialization of group generation (configurations) and social choice-based aggregation strategies

random.seed(time.time())

num_items = 50
group_size = 4
domains = ['tourist locations', 'movies', 'anon']

configuration_list = ['divergent', 'uniform', 'coalitional', 'minority']
configurations = {
    "coalitional": lambda: generate_coalitional_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "divergent":   lambda: generate_divergent_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "minority":    lambda: generate_minorty_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "uniform":     lambda: generate_uniform_conf(n=group_size, m=num_items, r=100, options=ITEMS),
}

strategies = {
    "ADD": lambda df: ADD(df),
    "MAJ": lambda df: MAJ_from_df(df),
    "LMS": lambda df: LMS(df),
    "MPL": lambda df: MPL(df),
    "APP": lambda df: APP(df, threshold=60),
    #"FAI": lambda: FAI(result),
    "BORDA": lambda df: BORDA_from_df(df),
    "AWM": lambda df: AWM(df, threshold=35),
}

In [23]:
import pickle
import pandas as pd

with open('real_dataset/group_composition.pkl', 'rb') as f:
    group_dict = pickle.load(f)

choices_df = pd.read_csv('real_dataset/group_choices.csv')
ratings_df = pd.read_csv('real_dataset/ratings.csv')

for group_id, info in group_dict.items():
    
    info['group_id'] = group_id

    group_choices = choices_df[choices_df['group_id'] == group_id]

    sorted_choices = group_choices.sort_values(by='rank')['item'].tolist()
    
    info['group_choices'] = sorted_choices
    
    members = info['group_members']  # e.g., [26323, 42775, 41651, 32327]
    
    group_ratings_df = ratings_df[ratings_df['user'].isin(members)]
    
    unique_items = group_ratings_df['item'].unique()
    
    formatted_ratings = {}
    
    for item in unique_items:
        item_ratings_list = []
        
        for member in members:
            match = group_ratings_df[(group_ratings_df['user'] == member) & (group_ratings_df['item'] == item)]
            
            if not match.empty:
                rating_value = match['rating'].values[0]
                item_ratings_list.append(int(rating_value))
            else:
                item_ratings_list.append(None) 
                
        formatted_ratings[item] = item_ratings_list
        
    info['member_ratings'] = formatted_ratings

import pprint
pprint.pprint(group_dict[1])

{'group_choices': ['D5', 'D3'],
 'group_id': 1,
 'group_members': [26323, 42775, 41651, 32327],
 'group_similarity': 'divergent',
 'group_size': 4,
 'member_ratings': {'D1': [6, 4, 10, 4],
                    'D10': [2, 6, 10, 4],
                    'D2': [4, 8, 10, 8],
                    'D3': [4, 8, 10, 4],
                    'D4': [2, 8, 10, 4],
                    'D5': [8, 10, 10, 6],
                    'D6': [6, 8, 10, 10],
                    'D7': [2, 10, 10, 6],
                    'D8': [4, 10, 10, 10],
                    'D9': [6, 8, 10, 8]}}


In [ ]:
import os
import json
import re
import time
import pandas as pd

file_path = 'results-realdata.csv'
completed = set()

# Adjusted columns: removed aggregation strategies, kept core metadata
columns = [
    'groupID', 'group_size', 'domain', 'llm', 
    'recommendation', 'explanation', 'group'
]

if os.path.exists(file_path):
    df_results = pd.read_csv(file_path)
    completed = set(
        zip(
            df_results['groupID'],
            df_results['domain'],
            df_results['llm']
        )
    )
else:
    df_results = pd.DataFrame(columns=columns)

groups_data = list(group_dict.values()) if isinstance(group_dict, dict) else group_dict

llms = ["ministral-3:8b-cloud", "gpt-oss:20b-cloud", "gpt-oss:120b-cloud","mistralai/mistral-large-2512"] # "mistral-large-3:675b-cloud",

## error catchers ##
errorcount = 0
ers = []
###############
def parse_llm_json(text: str) -> dict:
    try:
        return json.loads(text)

    except json.JSONDecodeError:
        cleaned = re.sub(r"```(?:json)?", "", text).strip()

        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in LLM output")

        return json.loads(match.group())

for group_obj in groups_data:
    try:
        groupid = group_obj['group_id']
        group_size = group_obj['group_size']
        member_ratings = group_obj['member_ratings'] 
        
        domain = 'real_dataset_tourism' 
        
        for llm_name in llms:
            if (groupid, domain, llm_name) in completed:
                continue

            system_message = {
                'role': 'system',
                'content': f"""
You are tasked with making group recommendations based on the different preferences of the group members.
You need explain the process behind making the recommendation to the group in such a way that someone without recommender systems knowledge can understand.
The information you are provided contain the preferences of the group. Every candidate item for recommendation has a rating from each user listed in the order by user (first rating from user1, second from user2 etc).
The rating is a scale from 0 to 10. For the recommendation, you simply mention the item identifier.
You make a recommendation to the group of users by providing a ranking of the items based on the recommendation approach you came up with.

Provide your answer as VALID JSON ONLY.
Do not use markdown or code fences.
Do not include newlines inside string values.
Use plain ASCII characters only.

Format:
{{
"recommendation": ["item1","item2","item3","item4","item5","item6","item7","item8","item9","item10"],
"explanation": "Short explanation of how you made the recommendation with no line breaks"
}}
"""
            }
            
            scenario = {
                'role': 'user',
                'content': f"""
The per-item ratings are presented below:
### BEGIN TABLE ###
{member_ratings}
### END TABLE ###

Think about the answer internally, but only output the final JSON object (containing recommendation ranking and explanation). Do not include any additional text or python code.
Return STRICT JSON. Do not use markdown.
"""
            }
            
            messages = [system_message, scenario]
            
            if 'gpt' in llm_name.lower(): 
                resp = client_ol.chat(llm_name, messages=messages, options={"temperature": 0.5})
                out = resp['message']['content']
            else:
                resp = client.chat.completions.create(
                    messages=messages,
                    model=llm_name,
                    stream=False,
                    max_completion_tokens=1000,
                    temperature=0.5,
                )
                out = resp.choices[0].message.content

            if "<think>" in out or "</think>" in out:
                out = re.sub(r'^.*?</think>', '', out, flags=re.DOTALL).strip()
                
            out = parse_llm_json(out)
            recommendation = out['recommendation']
            explanation = out['explanation']

            # Prepare the simplified row
            new_row = pd.DataFrame([{
                'groupID': groupid,
                'group_size': group_size,
                'domain': domain,
                'llm': llm_name,
                'recommendation': recommendation,
                'explanation': explanation,
                'group': str(member_ratings),
            }])

            if df_results.empty:
                df_results = new_row
            else:
                df_results = pd.concat([df_results, new_row], ignore_index=True)

            df_results.to_csv(file_path, index=False)
            time.sleep(2)

    except Exception as e: # Broadened catch to catch general errors or your custom RateLimitError
        errorcount += 1
        msg = str(e)
        ers.append(msg)
        print(f"Error encountered: {msg}")
        if errorcount > 5:
            print('Time to stop. Error count above 5')
            print(ers)
            break
        if "high traffic" in msg.lower() or "queue_exceeded" in msg.lower() or "limit" in msg.lower():
            print("High traffic detected.")
            time.sleep(20)
            continue

In [22]:
import os
import ast
import json
import pandas as pd

input_file = 'results-realdata.csv'
output_file = 'results-realdata-with-strategies3.csv'

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Could not find the input file: {input_file}")

df_results = pd.read_csv(input_file)

strategies = {
    "ADD": lambda df: ADD(df),
    "MAJ": lambda df: MAJ_from_df(df),
    "LMS": lambda df: LMS(df),
    "MPL": lambda df: MPL(df),
    "APP": lambda df: APP(df, threshold=6),       # Adjusted threshold to 6 since scale is 0-10
    "BORDA": lambda df: BORDA_from_df(df),
    "AWM": lambda df: AWM(df, threshold=3.5),     # Adjusted threshold to 3.5 since scale is 0-10
}

strategy_keys = list(strategies.keys())

strategy_lists = {s: [] for s in strategy_keys}

print(f"Processing {len(df_results)} rows from {input_file}...")

for idx, row in df_results.iterrows():
    try:

        group_dict = ast.literal_eval(row['group'])
        
        rating_rows = []
        for item, ratings in group_dict.items():
            for r in ratings:
                if isinstance(r, list):
                    rating_rows.extend([{"item": item, "rating": v} for v in r])
                else:
                    rating_rows.append({"item": item, "rating": r})
        
        rating_df = pd.DataFrame(rating_rows)
        
        for s in strategy_keys:
            try:
                ranking = strategies[s](rating_df)
                strategy_lists[s].append(ranking)
            except Exception as strategy_err:
                print(f"Error calculating strategy {s} for row index {idx}: {strategy_err}")
                strategy_lists[s].append(None)
                
    except Exception as parse_err:
        print(f"Error parsing group dictionary at row index {idx}: {parse_err}")
        for s in strategy_keys:
            strategy_lists[s].append(None)

for s in strategy_keys:
    df_results[s] = strategy_lists[s]

df_results.to_csv(output_file, index=False)
print(f"Successfully processed and saved results to {output_file}")

Processing 319 rows from results-realdata.csv...
Successfully processed and saved results to results-realdata-with-strategies3.csv


In [ ]:
import pickle
import pandas as pd

with open('real_dataset/group_composition.pkl', 'rb') as f:
    group_dict = pickle.load(f)

choices_df = pd.read_csv('real_dataset/group_choices.csv')
ratings_df = pd.read_csv('real_dataset/ratings.csv')

for group_id, info in group_dict.items():
    
    info['group_id'] = group_id

    group_choices = choices_df[choices_df['group_id'] == group_id]

    sorted_choices = group_choices.sort_values(by='rank')['item'].tolist()
    
    info['group_choices'] = sorted_choices
    
    members = info['group_members']  # e.g., [26323, 42775, 41651, 32327]
    
    group_ratings_df = ratings_df[ratings_df['user'].isin(members)]
    
    unique_items = group_ratings_df['item'].unique()
    
    formatted_ratings = {}
    
    for item in unique_items:
        item_ratings_list = []
        
        for member in members:
            match = group_ratings_df[(group_ratings_df['user'] == member) & (group_ratings_df['item'] == item)]
            
            if not match.empty:
                rating_value = match['rating'].values[0]
                item_ratings_list.append(int(rating_value))
            else:
                item_ratings_list.append(None) 
                
        formatted_ratings[item] = item_ratings_list
        
    info['member_ratings'] = formatted_ratings

import pprint
pprint.pprint(group_dict[1])

In [24]:
## ADDING GROUP CHOICES BACK TO THE DATASET
file_path = 'results-realdata-with-strategies3.csv'
df_results = pd.read_csv(file_path)

groups_list = list(group_dict.values()) if isinstance(group_dict, dict) else group_dict

choices_map = {g['group_id']: g['group_choices'] for g in groups_list if 'group_id' in g}
config_map = {g['group_id']: g['group_similarity'] for g in groups_list if 'group_id' in g}

df_results['group_choices'] = df_results['groupID'].map(choices_map)
df_results['configuration'] = df_results['groupID'].map(config_map)

df_results.to_csv(file_path, index=False)


In [25]:
import ast
import numpy as np
import pandas as pd
import collections as c

### Helper functions (adjusted)

def dcg_at_k(relevance_scores, k=10):
    relevance_scores = np.array(relevance_scores)[:k]
    return np.sum(relevance_scores / np.log2(np.arange(2, len(relevance_scores) + 2)))

def ndcg_at_k(predicted_order, gold_order, k=10, binary_relevance=False):
    if not isinstance(predicted_order, list) or not isinstance(gold_order, list):
        return 0.0
    if len(predicted_order) == 0 or len(gold_order) == 0:
        return 0.0

    if binary_relevance:
        gold_set = set(gold_order)
        predicted_relevance = [1 if item in gold_set else 0 for item in predicted_order]
        total_hits = min(len(gold_set), k)
        ideal_relevance = [1] * total_hits
    else:
        relevance_map = {item: len(gold_order) - i for i, item in enumerate(gold_order)}
        predicted_relevance = [relevance_map.get(item, 0) for item in predicted_order]
        ideal_relevance = sorted([relevance_map[item] for item in gold_order], reverse=True)
    
    dcg = dcg_at_k(predicted_relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg

def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return None
    return x

df_results['recommendation'] = df_results['recommendation'].apply(safe_literal_eval)
df_results['group_choices'] = df_results['group_choices'].apply(safe_literal_eval)

strategies_list = ["ADD", "APP", "LMS", "MPL", "MAJ", "BORDA", 'AWM']

for strategy in strategies_list:
    df_results[strategy] = df_results[strategy].apply(safe_literal_eval)
    
    llm_ndcg_col = f"{strategy}_LLM_ndcg2"
    df_results[llm_ndcg_col] = df_results.apply(
        lambda row: ndcg_at_k(row["recommendation"], row[strategy], k=2, binary_relevance=False),
        axis=1
    )
    
    choices_ndcg_col = f"{strategy}_Group_ndcg2"
    df_results[choices_ndcg_col] = df_results.apply(
        lambda row: ndcg_at_k(row["group_choices"], row[strategy], k=2, binary_relevance=False),
        axis=1
    )

df_results

,groupID,group_size,domain,llm,recommendation,explanation,group,ADD,MAJ,LMS,...,LMS_LLM_ndcg2,LMS_Group_ndcg2,MPL_LLM_ndcg2,MPL_Group_ndcg2,MAJ_LLM_ndcg2,MAJ_Group_ndcg2,BORDA_LLM_ndcg2,BORDA_Group_ndcg2,AWM_LLM_ndcg2,AWM_Group_ndcg2
0,1,4,domains,ministral-3:8b-cloud,"[D6, D8, D5, D2, D9, D3, D7, D1, D4, D10]",I calculated the average rating for each item ...,"{'D1': [6, 4, 10, 4], 'D2': [4, 8, 10, 8], 'D3...","[D5, D6, D8, D9, D2, D7, D3, D1, D4, D10]","[D5, D8, D6, D7, D1, D2, D3, D4, D9, D10]","[D5, D6, D9, D1, D2, D3, D8, D10, D4, D7]",...,0.735008,0.839032,0.335613,0.600605,0.872436,0.798790,0.895976,0.798790,0.848786,0.766010
1,10,4,domains,ministral-3:8b-cloud,"[D6, D9, D8, D3, D4, D7, D10, D2, D5, D1]",I averaged each item's ratings across all user...,"{'D1': [2, 6, 8, 8], 'D2': [2, 8, 4, 10], 'D3'...","[D6, D8, D9, D1, D10, D2, D3, D4, D5, D7]","[D6, D8, D2, D5, D9, D1, D3, D4, D7, D10]","[D8, D9, D1, D10, D2, D3, D4, D5, D6, D7]",...,0.489743,0.647685,0.751710,0.375855,0.879274,0.224750,0.839032,0.704629,0.239812,0.000000
2,11,4,domains,ministral-3:8b-cloud,"[D2, D3, D4, D10, D9, D1, D8, D6, D5, D7]",To find the best group recommendation I averag...,"{'D1': [10, 8, 3, 1], 'D2': [6, 10, 10, 5], 'D...","[D2, D10, D3, D4, D1, D9, D8, D7, D6, D5]","[D2, D1, D7, D3, D4, D5, D6, D8, D9, D10]","[D10, D2, D3, D4, D8, D9, D1, D5, D6, D7]",...,0.895976,0.855734,0.815492,0.694766,0.919516,0.879274,1.000000,0.919516,0.892932,0.785864
3,1,4,real_dataset_tourism,ministral-3:8b-cloud,"[D6, D8, D5, D9, D2, D3, D1, D7, D4, D10]",To find the best group recommendation I averag...,"{'D1': [6, 4, 10, 4], 'D2': [4, 8, 10, 8], 'D3...","[D5, D6, D8, D9, D2, D7, D3, D1, D4, D10]","[D5, D8, D6, D7, D1, D2, D3, D4, D9, D10]","[D5, D6, D9, D1, D2, D3, D8, D10, D4, D7]",...,0.735008,0.839032,0.335613,0.600605,0.872436,0.798790,0.895976,0.798790,0.848786,0.766010
4,1,4,real_dataset_tourism,gpt-oss:20b-cloud,"[D5, D6, D8, D9, D2, D7, D3, D1, D4, D10]",We calculated each item's average rating acros...,"{'D1': [6, 4, 10, 4], 'D2': [4, 8, 10, 8], 'D3...","[D5, D6, D8, D9, D2, D7, D3, D1, D4, D10]","[D5, D8, D6, D7, D1, D2, D3, D4, D9, D10]","[D5, D6, D9, D1, D2, D3, D8, D10, D4, D7]",...,1.000000,0.839032,0.479879,0.600605,0.959758,0.798790,1.000000,0.798790,1.000000,0.766010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,84,2,real_dataset_tourism,mistralai/mistral-large-2512,"[D1, D10, D5, D7, D8, D3, D4, D6, D2, D9]",I ranked the items by first looking at the low...,"{'D5': [10, 10], 'D7': [10, 9], 'D2': [9, 8], ...","[D1, D10, D5, D7, D8, D3, D4, D6, D2, D9]","[D5, D1, D10, D7, D9, D3, D4, D6, D8, D2]","[D1, D10, D5, D7, D8, D2, D3, D4, D6, D9]",...,1.000000,0.630983,1.000000,0.664387,0.895976,0.798790,0.895976,0.798790,1.000000,0.671225
315,85,2,real_dataset_tourism,mistralai/mistral-large-2512,"[D2, D3, D4, D7, D8, D5, D1, D9, D10, D6]",I balanced the group's preferences by averagin...,"{'D5': [8, 4], 'D7': [6, 7], 'D2': [8, 9], 'D9...","[D2, D3, D4, D8, D10, D7, D5, D6, D1, D9]","[D2, D10, D6, D5, D7, D9, D3, D1, D4, D8]","[D2, D3, D4, D7, D8, D5, D1, D10, D9, D6]",...,1.000000,0.735008,0.791952,0.647685,0.798790,0.375855,1.000000,0.654523,1.000000,0.546171
316,86,4,real_dataset_tourism,mistralai/mistral-large-2512,"[D8, D4, D3, D10, D1, D9, D2, D5, D7, D6]",I ranked the items by first calculating the av...,"{'D5': [8, 9, 6, 10], 'D7': [6, 8, 4, 6], 'D2'...","[D8, D3, D5, D4, D1, D9, D10, D2, D6, D7]","[D8, D5, D1, D4, D2, D3, D10, D6, D7, D9]","[D3, D8, D9, D1, D4, D5, D10, D2, D7, D6]",...,0.815492,0.536823,0.536823,0.976460,0.919516,0.577065,0.879274,0.600605,0.910172,0.438133
317,87,2,real_dataset_tourism,mistralai/mistral-large-2512,"[D1, D4, D10, D6, D3, D5, D7, D8, D2, D9]",I ranked the items by first looking at the ave...,"{'D5': [8, 5], 'D7': [8, 5], 'D2': [8, 2], 'D9...","[D1, D4, D10, D6, D3, D5, D7, D8, D2, D9]","[D1, D4, D10, D6, D5, D7, D2, D9, D3, D8]","[D1, D4, D10, D3, D6, D5, D7, D

In [26]:
## Adjust k for ndcg@2 or ndcg@10 (both reported in Section 5.4)


llm_vs_choices_col = "LLM_vs_Choices_ndcg2"
df_results[llm_vs_choices_col] = df_results.apply(
    lambda row: ndcg_at_k(row["recommendation"], row["group_choices"], k=10, binary_relevance=False),
    axis=1
)

In [ ]:
# TABLE FROM PAPER, @10
display(df_results[[ 'llm',"LLM_vs_Choices_ndcg2"]].groupby(['llm', ]).mean())


,LLM_vs_Choices_ndcg2
llm,
gpt-oss:120b-cloud,0.733015
gpt-oss:20b-cloud,0.733666
ministral-3:8b-cloud,0.692311
mistralai/mistral-large-2512,0.718139


In [28]:
llm_vs_choices_col = "LLM_vs_Choices_ndcg2"
df_results[llm_vs_choices_col] = df_results.apply(
    lambda row: ndcg_at_k(row["recommendation"], row["group_choices"], k=2, binary_relevance=False),
    axis=1
)
# TABLE FROM PAPER, @2
display(df_results[[ 'llm',"LLM_vs_Choices_ndcg2"]].groupby(['llm', ]).mean())

,LLM_vs_Choices_ndcg2
llm,
gpt-oss:120b-cloud,0.509463
gpt-oss:20b-cloud,0.519567
ministral-3:8b-cloud,0.438031
mistralai/mistral-large-2512,0.496026
